# Zavran AI — Zaroon Voice Cloning, Audio Synthesis & Strict Guardrails Notebook

This notebook verifies:
1. **Master Recording Validation**: Checks `D:\Zevaran\Zaroon.mp3` source file attributes.
2. **Voice Cloning & TTS Synthesis**: Synthesizes speech using Zaroon's exclusive cloned voice model.
3. **In-Notebook Audio Playback**: Plays generated audio waveforms directly in Jupyter.
4. **Persona Isolation Guardrails**: Enforces that Zaroon uses only his cloned voice and rejects unauthorized personas.
5. **State Machine & Silence Detection**: Demonstrates non-overlapping turn-taking and silence transitions.

In [ ]:
import os
import sys
import json
import base64
from pathlib import Path
from IPython.display import Audio, display

# Setup Workspace Path
WORKSPACE_DIR = Path(".").resolve()
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

from backend.config import settings, load_env_file
from backend.providers.tts_provider import ZaroonTTSProvider, get_tts_provider

print("[*] Workspace initialized at:", WORKSPACE_DIR)

## 1. Verify Master Source Audio File (`Zaroon.mp3`)

In [ ]:
audio_file = WORKSPACE_DIR / "Zaroon.mp3"
assert audio_file.exists(), f"Zaroon.mp3 not found at {audio_file}"

size_bytes = audio_file.stat().st_size
print(f"[SUCCESS] Zaroon.mp3 found!")
print(f"  Path: {audio_file}")
print(f"  Size: {size_bytes:,} bytes (~{size_bytes / 1024:.1f} KB)")

# Play the master voice recording
display(Audio(str(audio_file)))

## 2. Test Zaroon Voice Synthesis with Technical Interview Questions

In [ ]:
import asyncio

provider = ZaroonTTSProvider()
sample_text = (
    "Hello and welcome to your technical interview with Zavran AI. "
    "I am Zaroon. Could you walk me through how you architect low-latency "
    "streaming pipelines with FastAPI and PostgreSQL?"
)

print("[*] Synthesizing sample text with Zaroon's voice ID:", provider.voice_id or "(Mock / Configured ID)")
response = asyncio.run(provider.synthesize_speech(sample_text))

print("[*] Status:", response.get("status"))
print("[*] Persona:", response.get("persona"))

if response.get("status") == "success" and response.get("audio_base64"):
    audio_bytes = base64.b64decode(response["audio_base64"])
    print(f"[SUCCESS] Synthesized {len(audio_bytes):,} bytes of audio.")
    display(Audio(data=audio_bytes, format="wav", autoplay=True))
else:
    print("[NOTE] Live synthesis returned:", response.get("error", response.get("technical_error")))

## 3. Strict Persona Routing & Security Guardrails Test

In [ ]:
# Guardrail Test 1: Zaroon routes strictly to ZaroonTTSProvider
zaroon_p = get_tts_provider("Zaroon")
assert isinstance(zaroon_p, ZaroonTTSProvider), "Zaroon must route to ZaroonTTSProvider"
print("[PASS] Zaroon persona routing verified.")

# Guardrail Test 2: Other personas (Aarin, Soni) route to their respective providers
aarin_p = get_tts_provider("Aarin")
soni_p = get_tts_provider("Soni")
assert not isinstance(aarin_p, ZaroonTTSProvider), "Aarin must not use Zaroon's voice"
assert not isinstance(soni_p, ZaroonTTSProvider), "Soni must not use Zaroon's voice"
print("[PASS] Persona voice isolation verified.")

# Guardrail Test 3: Unauthorized / candidate persona is rejected
try:
    get_tts_provider("Candidate")
    print("[FAIL] Unauthorized persona was not rejected!")
except ValueError as e:
    print("[PASS] Unauthorized persona successfully blocked:", e)

## 4. Silence Detection & Turn-Taking Simulation

In [ ]:
class VoiceTurnSimulator:
    """Simulates Zaroon turn-taking with 3-second silence detection."""
    
    def __init__(self):
        self.state = "INTERVIEW_READY"
    
    def ai_speaks(self, question: str):
        self.state = "AI_SPEAKING"
        print(f"[ZAROON AUDIO] Speaking: '{question}'")
        self.state = "LISTENING"
        print("[CANDIDATE MIC] Active — Listening...")
    
    def candidate_speaks(self, transcript_chunk: str, pause_duration_seconds: float):
        print(f"  Candidate says: '{transcript_chunk}'")
        if pause_duration_seconds < 2.0:
            print(f"  [PAUSE {pause_duration_seconds}s] Short thinking pause — Zaroon waits silently.")
        elif pause_duration_seconds < 3.0:
            print(f"  [PAUSE {pause_duration_seconds}s] Possible continuation — keeping mic open.")
        else:
            print(f"  [SILENCE {pause_duration_seconds}s] Answer complete detected!")
            self.state = "ANSWER_COMPLETE"
            self.state = "EVALUATING"
            print("  [BACKEND] Evaluating transcript against expected answer model...")
            self.state = "NEXT_QUESTION"

sim = VoiceTurnSimulator()
sim.ai_speaks("How do you handle Redis cache invalidation in distributed services?")
sim.candidate_speaks("We use an event-driven cache invalidation pattern via Kafka topic listeners...", 1.5)
sim.candidate_speaks("...and we fallback to TTL expiration if event delivery lags.", 3.2)
print("[STATE MACHINE FINAL]", sim.state)
